[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Regression Metrics


## What you will be able to do

Measure a regression model's errors with R², mean absolute error, root mean squared error and mean
absolute percentage error, and look past the single number to where the errors fall.

## The idea

### The problem

The linear model of January temperature scores an R² of 0.888 on the stations it did not train on.
That is a good number, and it answers almost none of the questions someone would ask about the
model. How many degrees off is a typical prediction? What is the worst miss? Is the model wrong in
the same direction for coastal stations, or for mountain ones? A planner choosing a heater for a new
station wants degrees, an engineer wants the worst case, and a forecaster wants to know whether one
kind of station is always too warm.

A single number cannot answer all of those, and different numbers answer different ones. Some are in
the target's own units and some are not. Some treat a few large misses as worse than many small
ones. One reports errors as a percentage, which sounds natural and breaks on temperatures near zero.

### What a regression metric is

> A **regression metric** compares a model's predictions with the measured values of rows it did not
> train on, through the **residuals**, each measured value minus its prediction, and condenses them
> into one number. **R²** compares the squared residuals with the spread of the measured values,
> **mean absolute error** (MAE) averages their sizes in the target's units, **root mean squared
> error** (RMSE) is the square root of their squared average, and **mean absolute percentage error**
> (MAPE) averages each residual's size as a share of its measured value.

### Why it works that way

R² is relative. It is 1 for perfect predictions and 0 for predicting the mean of the measured values
every time, and nothing stops it going below 0 for a model worse than that. Because it divides by
the spread of the rows being scored, the same errors give a lower R² on rows that vary less, which
the **Cross-Validation** notebook saw in folds sorted by January mean.

MAE is in degrees, so it reads directly as a typical miss. RMSE squares the residuals before
averaging them, so a few large misses weigh more, and it is always at least as large as MAE. For
errors shaped like a bell curve, RMSE is about 1.25 times MAE, and a ratio well above that is a sign
that some misses are much larger than the rest. MAPE divides each residual by its measured value, so
a station with a January mean of 0.1 degrees turns a two-degree miss into thousands of percent.

Every one of them averages over all the rows, so none says where the errors are. That takes the
residuals themselves, grouped by something you know about the rows.

### Where this shows up

The **Cross-Validation** notebook scored models with `"r2"` and `"neg_mean_absolute_error"`, and
those names map onto the functions here. The **A Complete Workflow** notebook reports its final
model with them, and the statistics behind them are in the Visual Handbook.

### What this notebook covers

- three models on the same held-back stations, the mean of the training stations among them
- R², MAE, RMSE, the largest error and the median error, side by side
- the residuals, the largest misses, and the errors grouped by terrain
- MAPE, and the stations that make it large
- four errors, three of which return a number without a warning

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

measured = [2.0, -1.0, 0.5, -3.0]
predicted = [1.5, -1.0, 1.0, -2.0]
print(round(r2_score(measured, predicted), 3))
print(round(mean_absolute_error(measured, predicted), 3))
print(round(root_mean_squared_error(measured, predicted), 3))
```

```
0.89
0.5
0.612
```

The residuals are 0.5, 0, -0.5 and -1. Their average size is 0.5, the MAE. RMSE, 0.612, is larger,
because the one miss of a whole degree counts more once squared. R² says the predictions account for
89 percent of the variation in the four measured values.

## Setup

Two libraries for the data, scikit-learn, three models and the metric functions.

- `numpy` and `pandas` build the stations and group the residuals, and the cell prints
  scikit-learn's version
- `DummyRegressor` predicts the mean of the training stations for every station, the baseline any
  model has to beat
- `LinearRegression` and `RandomForestRegressor` are the two models of January temperature
- `train_test_split` holds a quarter of the stations back, and `cross_val_predict` with `KFold`
  predicts every station from folds that left it out, in the third Common error
- `r2_score`, `mean_absolute_error`, `root_mean_squared_error`, `max_error`,
  `median_absolute_error`, `mean_absolute_percentage_error` and `mean_squared_error` are the
  metrics, each a function of the measured values and the predictions, in that order

`station_network` makes a network of 1,200 weather stations in Norway. The stations are made up, and
a generator started from a fixed seed draws them, so the table is the same on every run. Each row
has a station's latitude, longitude, elevation in meters, distance from the coast in kilometers and
the year it was installed, its terrain and owner as text, and three things to predict, of which this
notebook uses one: the January mean in degrees Celsius. `FEATURES` names the five numeric columns
the models use.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (max_error, mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, median_absolute_error, r2_score, root_mean_squared_error)
from sklearn.model_selection import KFold, cross_val_predict, train_test_split

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### Models scored on the same held-back stations

A quarter of the stations is held back, and three models are fitted on the rest. `DummyRegressor`
learns nothing but the training stations' mean:

In [2]:
X = network[FEATURES]
y = network["jan_mean"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

models = {
    "mean only": DummyRegressor(),
    "linear": LinearRegression(),
    "forest": RandomForestRegressor(random_state=0),
}
predictions = {name: model.fit(X_train, y_train).predict(X_test) for name, model in models.items()}

print(len(y_test), "held-back stations")

300 held-back stations


### The metrics side by side

Each metric takes the measured values first and the predictions second:

In [3]:
table = pd.DataFrame({
    name: {
        "R²": r2_score(y_test, predicted),
        "MAE": mean_absolute_error(y_test, predicted),
        "RMSE": root_mean_squared_error(y_test, predicted),
                "RMSE / MAE": root_mean_squared_error(y_test, predicted) / mean_absolute_error(y_test, predicted),
        "largest error": max_error(y_test, predicted),
        "median error": median_absolute_error(y_test, predicted),
    }
    for name, predicted in predictions.items()
}).T.round(3)
table

,R²,MAE,RMSE,RMSE / MAE,largest error,median error
mean only,-0.013,2.330,3.011,1.292,8.949,1.849
linear,0.888,0.784,1.000,1.275,3.276,0.638
forest,0.894,0.752,0.972,1.294,3.239,0.578


The mean-only model's R² is -0.013, close to 0 and slightly below it, because the training stations'
mean is not quite the held-back stations' mean. Its MAE says that guessing the mean misses January
by 2.33 degrees on average. The linear model is off by 0.784 degrees on average, and the forest by
0.752, so on these 300 stations the forest is a little better on every number. Over five folds, the
**Cross-Validation** notebook found the two within 0.011 of each other in every fold. For the two
models RMSE is 1.28 and 1.29 times MAE, close to the 1.25 that errors shaped like a bell curve give,
so the gap is no sign of a few unusual misses. The largest single miss, just over three degrees, is
a little over three times the RMSE. The median error, the middle of the absolute residuals, is
smaller still, since half the stations are predicted within about 0.6 degrees.

### The residuals themselves

A residual is the measured value minus the prediction, so a positive one means the station was
milder than predicted:

In [4]:
residuals = y_test - predictions["linear"]

print("mean residual:", round(residuals.mean(), 3), "| standard deviation:", round(residuals.std(), 3))
print("stations missed by more than 2 degrees:", int((residuals.abs() > 2).sum()), "of", len(residuals))

largest = residuals.abs().nlargest(3).index
network.loc[largest, ["station", "terrain", "elevation_m", "coast_km", "jan_mean"]].assign(
    predicted=predictions["linear"][y_test.index.get_indexer(largest)].round(1)
)

mean residual: -0.124 | standard deviation: 0.994
stations missed by more than 2 degrees: 16 of 300


,station,terrain,elevation_m,coast_km,jan_mean,predicted
867,ST0868,inland,492,76.9,-5.1,-8.4
533,ST0534,inland,323,4.8,1.7,-1.1
1015,ST1016,coastal,153,0.8,-3.7,-6.5


The residuals average -0.124, a little below zero, so on these stations the model runs slightly warm
overall. Sixteen of the 300 stations are missed by more than two degrees. All three of the largest
misses were predicted colder than they were, by about three degrees, two inland and one on the
coast, and the table alone does not say why, which is what grouping the residuals helps with.

### Errors grouped by terrain

Grouping the residuals by something known about each station shows where the model goes wrong:

In [5]:
terrain = network.loc[y_test.index, "terrain"]

by_terrain = pd.DataFrame({
    "stations": terrain.value_counts(),
    "linear bias": residuals.groupby(terrain).mean(),
    "linear MAE": residuals.abs().groupby(terrain).mean(),
    "forest bias": (y_test - predictions["forest"]).groupby(terrain).mean(),
        "forest MAE": (y_test - predictions["forest"]).abs().groupby(terrain).mean(),
    "linear share of bias": residuals.groupby(terrain).sum() / len(residuals),
}).round(2)
by_terrain

,stations,linear bias,linear MAE,forest bias,forest MAE,linear share of bias
terrain,,,,,,
coastal,34,0.88,1.10,0.18,0.90,0.10
fjord,97,-0.01,0.56,-0.02,0.62,-0.00
inland,151,-0.35,0.84,-0.21,0.74,-0.18
mountain,18,-0.70,0.96,-1.27,1.33,-0.04


The bias is the mean residual. The linear model predicts the 34 coastal stations 0.88 degrees too
cold on average and the 18 mountain stations 0.7 degrees too warm, while fjord stations come out
almost exactly right. The overall numbers hid both. The last column is each terrain's part of the
overall mean residual, and the mean residual, -0.124, is mostly the 151 inland stations' -0.18,
partly offset by the coastal stations' errors of the opposite sign. The MAE, 0.784, mixes the
coastal stations' 1.1 degrees with the fjord stations' 0.56. The forest's coastal bias is smaller,
0.18, and its mountain bias larger, -1.27, on only 18 stations, so neither model is better
everywhere.

### MAPE on temperatures near zero

MAPE reports each residual as a share of its measured value. `mean_absolute_percentage_error`
returns that share as a fraction, so 0.337 means 33.7 percent:

In [6]:
percent = (residuals.abs() / y_test.abs() * 100)

print("MAPE:", round(mean_absolute_percentage_error(y_test, predictions["linear"]), 3))
print("median percentage error:", round(percent.median(), 1))

pd.DataFrame({
    "station": network.loc[y_test.index, "station"],
    "measured": y_test,
    "predicted": predictions["linear"].round(1),
    "percent off": percent.round(0),
}).nlargest(4, "percent off")

MAPE: 0.337
median percentage error: 13.4


,station,measured,predicted,percent off
443,ST0444,-0.1,-2.2,2086.0
1111,ST1112,-0.3,-2.2,648.0
702,ST0703,0.3,-1.6,630.0
1179,ST1180,-0.3,-2.2,628.0


A MAPE of 33.7 percent sounds far worse than an error of 0.784 degrees, for the same predictions.
The median station is off by 13.4 percent, and the average is pulled up by stations whose January
mean is close to zero. Station ST0444 measured -0.1 degrees and was predicted at -2.2, a miss of
about two degrees and of 2,086 percent. MAPE suits a target that is measured from a true zero and
stays well away from it, such as a price. A January mean in Celsius does neither. Its zero is the
point at which water freezes, and many stations sit close to it.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/07-regression-metrics-solutions.ipynb).

**1.** Print the forest's MAE and RMSE on the held-back stations, rounded to three places, and the difference between them.

In [7]:
# your code here

**2.** Print the linear model's R² and MAE on the held-back stations north of latitude 68, and how many of them there are.

In [8]:
# your code here

**3.** Print the linear model's mean residual on the held-back stations for each `owner`, rounded to two places.

In [9]:
# your code here

**4.** Fit `DummyRegressor(strategy="median")` on the training stations and print its MAE on the held-back stations beside the mean-only model's.

In [10]:
# your code here

**5.** Show the five held-back stations the forest misses most, with their terrain, measured value and residual.

In [11]:
# your code here

## Common errors

### TypeError: got an unexpected keyword argument 'squared'

In [12]:
mean_squared_error(y_test, predictions["linear"], squared=False)

TypeError: got an unexpected keyword argument 'squared'

Older tutorials get RMSE this way, from `mean_squared_error` with `squared=False`. scikit-learn 1.6
removed the argument, and `root_mean_squared_error` is the function that replaced it. Without the
argument, `mean_squared_error` is the squared error itself, in degrees squared:

In [13]:
print("MSE: ", round(mean_squared_error(y_test, predictions["linear"]), 3))
print("RMSE:", round(root_mean_squared_error(y_test, predictions["linear"]), 3))

MSE:  0.999
RMSE: 1.0


### No error, and an R² below zero

In [14]:
three_too_warm = np.full(len(y_test), y_train.mean() + 3)

print(round(r2_score(y_test, three_too_warm), 3))

-1.245


A "model" that predicts the training mean plus three degrees for every station scores -1.245. R² has
no lower limit, and a negative one means the predictions are worse than predicting the measured
values' own mean. A negative R² on held-back rows is worth checking for before anything else. The
model may have learned something wrong, or the rows may differ from its training rows, or they may
vary so little that ordinary errors outweigh their spread, as in the sorted folds of the
**Cross-Validation** notebook.

### No error, and a MAPE of 4.03e+12

Predicting every station from folds that left it out gives a prediction for all 1,200:

In [15]:
every_station = cross_val_predict(LinearRegression(), X, y, cv=KFold(5, shuffle=True, random_state=0))

print("MAE:", round(mean_absolute_error(y, every_station), 3))
print(f"MAPE: {mean_absolute_percentage_error(y, every_station):.3g}")
print("stations with a January mean of exactly 0:", network.loc[y == 0, "station"].tolist())
print("what scikit-learn divides by in place of 0:", np.finfo(np.float64).eps)

MAE: 0.814
MAPE: 4.03e+12
stations with a January mean of exactly 0: ['ST0391']
what scikit-learn divides by in place of 0: 2.220446049250313e-16


The MAE, 0.814 degrees, is the mean of the five fold errors that the **Cross-Validation** notebook
printed for the same folds. The MAPE is about four trillion as a fraction, which is 400 trillion
percent. One station, ST0391, has a January mean of exactly 0.0, and dividing by zero is undefined,
so scikit-learn divides that station's error by 2.2e-16 instead, and the one station's share swamps
every other. Nothing warns. Use MAE or RMSE for a target that can be zero.

### No error, and a different R² from swapped arguments

In [16]:
right_order = r2_score(y_test, predictions["linear"])
swapped = r2_score(predictions["linear"], y_test)

print("r2_score(measured, predicted):", round(right_order, 3))
print("r2_score(predicted, measured):", round(swapped, 3))
print("MAE either way:", round(mean_absolute_error(y_test, predictions["linear"]), 3),
      round(mean_absolute_error(predictions["linear"], y_test), 3))

r2_score(measured, predicted): 0.888
r2_score(predicted, measured): 0.875
MAE either way: 0.784 0.784


Every metric takes the measured values first. R² divides by the spread of whatever comes first, so
with the arguments swapped it divided by the spread of the predictions, and reported 0.875 instead
of 0.888. MAE gives the same answer either way, because the size of a difference does not depend on
its order, so a check with MAE does not catch the mistake. Pass the measured values first, or name
them, as `r2_score(y_true=..., y_pred=...)`.

## Recap

- Score a regressor on held-back rows with R², MAE and RMSE, and compare it with a model that
  predicts the mean
- Read MAE and RMSE in the target's units, and an RMSE well above 1.25 times MAE as a sign of a few
  large misses
- Group the residuals by something known about the rows to find where a model is biased
- Avoid MAPE for a target that can be at or near zero, and pass the measured values first

## What is next

The **Classification Metrics** notebook does the same for a yes-or-no answer: snow on 1 May, the
errors of each kind, and the threshold that trades one for the other.

---

&#8592; **Previous:** [Cross-Validation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/06-cross-validation.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Classification Metrics](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/08-classification-metrics.ipynb) &#8594;
